In [14]:
import sys
sys.path.append('../')
import UTILS.utils as utils

from PIL import Image
import pytesseract
from tqdm import tqdm
import fitz
from typing import Sequence

from google.cloud import vision
import google.generativeai as genai
from google.api_core.client_options import ClientOptions
from google.cloud import documentai



In [2]:
book = fitz.open(utils.FULL_TEXT_PDF)

# convert pdf of book pages to PNG for easier processing
def convert_full_book_to_png():
    for page_num in tqdm(range(book.page_count)):
        page = book.load_page(page_num)
        pix = page.get_pixmap(dpi=300)
        output = "page_" + str(page_num) + ".png"
        pix.save(utils.ALL_PAGES_PNG + output)

    book.close()

convert_full_book_to_png()

100%|██████████| 366/366 [01:49<00:00,  3.35it/s]


In [2]:
all_pages = utils.get_files_in_directory(utils.ALL_PAGES_PNG)

### OCR W/ TESSERACT

In [ ]:
# OCR using PyTesseract
def get_ocr_from_png_tesseract():
    for i in tqdm(range(utils.FIRST_PAGE_OCR, utils.LAST_PAGE_OCR)):
        image_path = utils.ALL_PAGES_PNG + "page_" + str(i) + ".png"

        text = pytesseract.image_to_string(Image.open(image_path))
        text_path = utils.ALL_PAGES_TXT_TESSERACT + "page_" + str(i) + ".txt"
        utils.write_to_file(text_path, text, create=True)

# get_ocr_from_png_tesseract()

### OCR W/ GEMINI

In [ ]:
genai.configure(api_key=utils.API_KEY())
model = genai.GenerativeModel('gemini-1.5-flash')

In [4]:
prompt = """Provide an OCR of the following PNG. 
Please provide only the OCR without the page title and number, and without extra comments. 
Keep the formatting of the page."""

In [ ]:
def prompt_model(file, prompt, page_num, directory, create_file = True):
    response = model.generate_content(
    contents=[
        file,
        prompt])
    if create_file:
        f = open(directory + "page_" + page_num + ".txt", "x")
        f.write(response.text)
        f.close()
    return response.text

In [12]:
def get_OCR_all_pages():
    for i in tqdm(range(utils.FIRST_PAGE_OCR, utils.LAST_PAGE_OCR)):
        image_path = utils.ALL_PAGES_PNG + "page_" + str(i) + ".png"
        sample_file = genai.upload_file(path=image_path, display_name="file1")
        page_num = utils.get_page_number(image_path)
        prompt_model(sample_file, prompt, page_num, utils.ALL_PAGES_TXT_GEMINI, create_file=True)

#get_OCR_all_pages()

100%|██████████| 168/168 [18:42<00:00,  6.68s/it]


### OCR W/ CLOUD VISION

In [15]:
client = vision.ImageAnnotatorClient.from_service_account_json("../../moonchu-pdm-b06f514f47a3.json")

In [16]:
breaks = vision.TextAnnotation.DetectedBreak.BreakType
paragraphs = []
lines = [] 

# parse OCR with cloud vision AI
def get_and_parse_image_ocr(image_path): 
    paragraphs = [] # tracks all paragraphs
    lines = [] # tracks all lines
    with open(image_path, 'rb') as image:
        content = image.read()
        image = vision.Image(content=content)
        response = client.text_detection(image=image)
        texts = response.full_text_annotation
        for page in texts.pages: # for each page
            for block in page.blocks: # for each block
                for paragraph in block.paragraphs: # for each paragraph
                    para = "" # tracks current paragraph
                    line = "" # tracks current line
                    for word in paragraph.words: # for each word
                        for symbol in word.symbols: # for each symbol
                            line += symbol.text # add symbol to line

                            if symbol.property.detected_break.type == breaks.SPACE:
                                # if break type is space
                                line += ' ' # add space

                            if symbol.property.detected_break.type == breaks.EOL_SURE_SPACE:
                                # if break type is end of line
                                if line[-1] == "-":
                                    line = line[:-1] # if last symbol is hyphen, remove hyphen
                                else:
                                    line += ' ' # otherwise, add space
                                lines.append(line) # add line to list of lines + to paragraph
                                para += line
                                line = '' # re-initialise line

                            if symbol.property.detected_break.type == breaks.LINE_BREAK:
                                # if last symbol is line break, add line to list of lines + to current paragraph
                                lines.append(line)
                                para += line
                                line = ''

                    paragraphs.append(para)
    return (lines, paragraphs)


In [17]:
def get_ocr_from_png_vision():
    for i in tqdm(range(utils.FIRST_PAGE_OCR, utils.LAST_PAGE_OCR)):
        image_path = utils.ALL_PAGES_PNG + "page_" + str(i) + ".png"

        (lines, paragraphs) = get_and_parse_image_ocr(image_path)

        txt_path = "page_" + str(i) + ".txt"
        utils.write_to_file(utils.ALL_PAGES_TXT_VISION_LINES + txt_path, "\n".join(lines), True)
        utils.write_to_file(utils.ALL_PAGES_TXT_VISION_PARAGRAPHS + txt_path, "\n".join(paragraphs), True)

       
get_ocr_from_png_vision()

100%|██████████| 1/1 [00:00<00:00,  2.22it/s]


### OCR W/ DOCUMENT AI

In [18]:
# convert DocumentAI Layout to string
def layout_to_text(layout: documentai.Document.Page.Layout, text: str) -> str:
    # If a text segment spans several lines, it will
    # be stored in different text segments.
    return "".join(
        text[int(segment.start_index) : int(segment.end_index)]
        for segment in layout.text_anchor.text_segments
    )

# get array of paragraphs from DocumentAI
def get_paragraphs_array(
    paragraphs: Sequence[documentai.Document.Page.Paragraph], text: str
) -> None:
    paragraph_array = []
    for paragraph in paragraphs:
        paragraph_array.append(layout_to_text(paragraph.layout, text))
    return paragraph_array

In [19]:
PROJECT_ID = "moonchu-pdm"
LOCATION = "us" # Format is "us" or "eu"
PROCESSOR_ID = "648dabce08f346bb" # Create processor before running sample
MIME_TYPE = "image/png" # Refer to https://cloud.google.com/document-ai/docs/file-types for supported file types

docai_client = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com", credentials_file="../../moonchu-pdm-b06f514f47a3.json")
)
RESOURCE_NAME = docai_client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

In [20]:
# parse document OCR for 1 file
def get_and_parse_document_ocr(image_path):
    with open(image_path, "rb") as image:
        image_content = image.read()

        raw_document = documentai.RawDocument(content=image_content, mime_type=MIME_TYPE)
        request = documentai.ProcessRequest(name=RESOURCE_NAME, raw_document=raw_document)
        result = docai_client.process_document(request=request)

        document_object = result.document
        text = document_object.text
        for page in document_object.pages:
            return get_paragraphs_array(page.paragraphs, text)


In [ ]:
def get_ocr_from_png_document_ai():
     for i in tqdm(range(utils.FIRST_PAGE_OCR, utils.LAST_PAGE_OCR)):
        image_path = utils.ALL_PAGES_PNG + "page_" + str(i) + ".png"

        paragraphs_array = get_and_parse_document_ocr(image_path)

        txt_path = "page_" + str(i) + ".txt"
        utils.write_to_file(utils.ALL_PAGES_TXT_DOCUMENT + txt_path, "¶".join(paragraphs_array), True)

get_ocr_from_png_document_ai()